# Q2: CNN vs MLP: a fair comparison, a spatial-structure test, and local vs global structure

| Part | Experiment | Status |
|---|---|---|
| **(a)** | MLP-M vs CNN at equal parameters and an identical training budget; fairness criterion fixed before any result | models and runs **reused from Q1(a)** |
| **(b)** | Fixed global pixel permutation (same pixel values, spatial structure destroyed); both models retrained | runs **reused from Q1(a)**; new prediction, test and analysis |
| **(c)** | **Per-image random tile shuffling** at 16, 8 and 4 px: separates *local* spatial structure from *global* arrangement; errors and confidence analysed | **new in Q2** |

**What is reused and how.** The data split, normalisation, `MLP`/`CNN` classes, counters and training loop are copied
verbatim from the Q1 notebook (marked in each cell). Runs whose configuration already exists in Q1(a) (MLP-M and CNN on
clean and on permuted images) are **loaded from Q1's run cache** (`Q1_OUT_DIR/runs`) when it is available. If it is not
available, they are retrained here with identical code, seeds and settings and written to that same cache. Section 7
lists every reused item.

**Data protocol (same as Q1).** The validation split is used for all selection: learning rate, early-stopping epoch, and
the decision rules' thresholds, which are fixed in advance. The official test set is loaded only in Section 5, after every model
is final. For every condition, **the test images go through exactly the transformation applied to the training
images**: the same fixed permutation in (b), and the same tile size and shuffling procedure in (c).

**Runtime.** Use a GPU (Colab: *Runtime → Change runtime type → T4 GPU*). If the Q1 cache is present, Q2 only trains
the 30 new runs of part (c), which should take roughly an hour on a T4. Without the Q1 cache, add about 40 minutes.
Setting `TUNE_LR_C = False` reuses each model's clean-image learning rate in part (c) and saves about a third of that time.
On Colab, point `Q1_OUT_DIR` and `OUT_DIR` to Google Drive so that runs survive disconnects.

In [ ]:
# ===================== 0. CONFIGURATION =====================
ROLL_NUMBER = "ENTER_YOUR_ROLL_NUMBER"   # must be the same roll number as in Q1

DATA_ROOT  = "./data"
Q1_OUT_DIR = "./q1_outputs"   # Q1 results folder; cached Q1(a) runs found here are reused, not retrained
OUT_DIR    = "./q2_outputs"   # Q2 figures, tables and new runs

# These must equal the Q1 settings so the Q1(a) runs can be reused (they define the cache key)
N_TRAIN, N_VAL = 45_000, 5_000
CFG_A = dict(epochs=30, batch_size=128, weight_decay=0.05, schedule="cosine")
LR_GRID = [3e-4, 1e-3, 3e-3]
N_SEEDS = 3
TRAIN_EVAL_SUBSET = 5_000

# Q2(c)
TILE_SIZES = [16, 8, 4]       # tile side in pixels (2x2, 4x4 and 8x8 grids)
TUNE_LR_C = True              # True: same LR grid per condition, as in (a)/(b). False: reuse each model's clean LR

In [ ]:
# ---- REUSED VERBATIM from the Q1 notebook (§0 setup); introduced in Q1 ----
import os, math, time, json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from scipy import stats
from sklearn.model_selection import train_test_split
from IPython.display import display

warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")
pd.set_option("display.width", 200)

digits = "".join(ch for ch in str(ROLL_NUMBER) if ch.isdigit())
if not digits:
    raise ValueError("Set ROLL_NUMBER in the configuration cell first.")
SEED = int(digits)

def derive_seed(*tags):
    '''32-bit seed derived deterministically from the roll-number seed plus a tag,
    so every random choice in the notebook traces back to SEED.'''
    words = [SEED] + [int.from_bytes(str(t).encode(), "little") for t in tags]
    return int(np.random.SeedSequence(words).generate_state(1)[0])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True
os.makedirs(os.path.join(OUT_DIR, "runs"), exist_ok=True)

# Plot style: fixed-order colour-blind-checked categorical palette, recessive grid
PALETTE = ["#257de0", "#f47230", "#00b894", "#f2a600", "#ee7fac", "#008a17", "#6a4ce2", "#ec4647"]
GREY = "#7a7a7a"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.prop_cycle": plt.cycler(color=PALETTE), "axes.grid": True,
    "grid.color": "#e7e7e7", "grid.linewidth": 0.8, "axes.edgecolor": "#c2c2c2",
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "legend.frameon": False, "font.size": 10,
})
def savefig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, name))
    plt.show()

def pct(x):
    return f"{100 * x:.1f}%"

def pp(x):
    return f"{100 * x:+.1f} pp"

print(f"Roll number {ROLL_NUMBER!r} -> seed {SEED}")
print(f"device={DEVICE}  AMP={USE_AMP}  torch={torch.__version__}")
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. The full budget is sized for a GPU; on CPU expect many hours.")

In [ ]:
from sklearn.metrics import roc_auc_score
Q1_RUNS = os.path.join(Q1_OUT_DIR, "runs")
os.makedirs(Q1_RUNS, exist_ok=True)
n_q1 = len([f for f in os.listdir(Q1_RUNS) if f.endswith(".pt")])
print(f"Q1 run cache: {os.path.abspath(Q1_RUNS)} -> {n_q1} cached runs found")

def identity(x):          # defined in Q1's permutation cell; needed here before the reused training loop
    return x

def sd(x):
    return float(np.std(x, ddof=1)) if len(x) > 1 else float("nan")

def mean_sd(x, digits=3):
    return f"{np.mean(x):.{digits}f} ± {sd(x):.{digits}f}" if len(x) > 1 else f"{np.mean(x):.{digits}f}"

## 1. Data (reused from Q1 §1)
This uses the same stratified 45,000 / 5,000 split (`train_test_split`, `random_state` = the roll-number digits) and the same
per-channel normalisation computed on the training split. The second cell checks that the split is identical to the one Q1 saved.

In [ ]:
# ---- REUSED VERBATIM from the Q1 notebook (§1 data); introduced in Q1 ----
train_full = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=True)
X_all = np.asarray(train_full.data)                       # (50000, 32, 32, 3) uint8
y_all = np.asarray(train_full.targets, dtype=np.int64)
CLASSES = list(train_full.classes)

# sklearn needs a 32-bit integer; a roll number with more than 9-10 digits is mapped through SeedSequence
split_state = SEED if SEED < 2**32 else derive_seed("split")
train_idx, val_idx = train_test_split(np.arange(len(y_all)), train_size=N_TRAIN, test_size=N_VAL,
                                      stratify=y_all, random_state=split_state)
np.savez(os.path.join(OUT_DIR, "split_indices.npz"), train_idx=train_idx, val_idx=val_idx, seed=SEED)

split_counts = pd.DataFrame({"train": np.bincount(y_all[train_idx], minlength=10),
                             "validation": np.bincount(y_all[val_idx], minlength=10)}, index=CLASSES)
assert len(np.intersect1d(train_idx, val_idx)) == 0
assert split_counts["train"].sum() == N_TRAIN and split_counts["validation"].sum() == N_VAL
print(f"random_state used for the split: {split_state}")
display(split_counts.T)

# Per-channel normalisation from the training split only
_x = torch.from_numpy(X_all[train_idx]).float().div_(255)
CH_MEAN, CH_STD = _x.mean(dim=(0, 1, 2)), _x.std(dim=(0, 1, 2))
del _x
print("channel mean", CH_MEAN.numpy().round(4), " std", CH_STD.numpy().round(4))

def to_tensor(x_u8):
    '''uint8 NHWC -> normalised float32 NCHW.'''
    x = torch.from_numpy(np.ascontiguousarray(x_u8)).float().div_(255)
    return ((x - CH_MEAN) / CH_STD).permute(0, 3, 1, 2).contiguous()

# The whole split lives on the GPU: no DataLoader overhead, and no augmentation is used anywhere in Q1
Xtr, ytr = to_tensor(X_all[train_idx]).to(DEVICE), torch.from_numpy(y_all[train_idx]).to(DEVICE)
Xva, yva = to_tensor(X_all[val_idx]).to(DEVICE), torch.from_numpy(y_all[val_idx]).to(DEVICE)
print("train", tuple(Xtr.shape), " validation", tuple(Xva.shape))

In [ ]:
q1_split = os.path.join(Q1_OUT_DIR, "split_indices.npz")
if os.path.exists(q1_split):
    s = np.load(q1_split)
    assert np.array_equal(s["train_idx"], train_idx) and np.array_equal(s["val_idx"], val_idx), \
        "Split differs from Q1: check ROLL_NUMBER"
    print("Split is identical to the one saved by Q1.")
else:
    print("Q1 split file not found: the split above was regenerated with the same seed and procedure.")

## 2. Shared components (reused verbatim from Q1 §2)
These are `MLP`, `CNN`, `count_params`, `count_macs`, `mlp_param_count`, `width_matching`, `predict_logits`, `train_model` and helpers.

In [ ]:
# ---- REUSED VERBATIM from the Q1 notebook (§2 models and counters); introduced in Q1 ----
class MLP(nn.Module):
    '''Flatten -> [Linear -> BatchNorm1d -> ReLU] x depth -> Linear.'''
    def __init__(self, width, depth=3, in_dim=3 * 32 * 32, n_classes=10):
        super().__init__()
        layers, d = [nn.Flatten()], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.BatchNorm1d(width), nn.ReLU(inplace=True)]
            d = width
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)
        self.spec = "MLP 3072 -> " + " -> ".join([str(width)] * depth) + " -> 10 (Linear+BN+ReLU per hidden layer)"

    def forward(self, x):
        return self.net(x)


def conv_bn_relu(cin, cout):
    return [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]


class CNN(nn.Module):
    '''Six 3x3 conv layers (BN+ReLU) in three stages with 2x2 max-pooling, global average pooling, linear head.'''
    def __init__(self, ch=(32, 64, 128, 128, 256, 256), n_classes=10):
        super().__init__()
        c1, c2, c3, c4, c5, c6 = ch
        self.features = nn.Sequential(
            *conv_bn_relu(3, c1), *conv_bn_relu(c1, c2), nn.MaxPool2d(2),     # 32x32 -> 16x16
            *conv_bn_relu(c2, c3), *conv_bn_relu(c3, c4), nn.MaxPool2d(2),    # 16x16 -> 8x8
            *conv_bn_relu(c4, c5), *conv_bn_relu(c5, c6), nn.MaxPool2d(2))    # 8x8   -> 4x4
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c6, n_classes))
        self.spec = (f"CNN conv3x3 [{c1},{c2}] pool | [{c3},{c4}] pool | [{c5},{c6}] pool | GAP | "
                     f"Linear {c6}->10 (BN+ReLU after each conv)")

    def forward(self, x):
        return self.head(self.features(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_macs(model, input_shape=(1, 3, 32, 32)):
    '''Multiply-accumulates of conv and linear layers for one image (BN, ReLU and pooling ignored).'''
    macs = 0
    def conv_hook(m, inp, out):
        nonlocal macs
        macs += out.numel() * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
    def lin_hook(m, inp, out):
        nonlocal macs
        macs += out.numel() * m.in_features
    hooks = [m.register_forward_hook(conv_hook) for m in model.modules() if isinstance(m, nn.Conv2d)]
    hooks += [m.register_forward_hook(lin_hook) for m in model.modules() if isinstance(m, nn.Linear)]
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(input_shape, device=next(model.parameters()).device))
    model.train(was_training)
    for h in hooks:
        h.remove()
    return macs


def mlp_param_count(width, depth=3, in_dim=3072, n_classes=10):
    '''Closed form for MLP(width, depth): Linear weights+biases plus BatchNorm scale/shift.'''
    first = in_dim * width + width + 2 * width
    hidden = (depth - 1) * (width * width + width + 2 * width)
    return first + hidden + width * n_classes + n_classes


def width_matching(target, depth=3):
    '''MLP width whose parameter count is closest to `target`.'''
    return min(range(8, 8192), key=lambda w: abs(mlp_param_count(w, depth) - target))

In [ ]:
# ---- REUSED VERBATIM from the Q1 notebook (§2 training loop); introduced in Q1 ----
@torch.no_grad()
def predict_logits(model, X, input_fn=identity, bs=2000):
    model.eval()
    out = []
    for i in range(0, X.shape[0], bs):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(model(input_fn(X[i:i + bs])).float())
    return torch.cat(out)


def loss_acc(logits, y):
    return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()


def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def df_to_lists(df):
    return {c: df[c].astype(float).tolist() for c in df.columns}


def train_model(model, X, y, Xv, yv, *, epochs, lr, weight_decay, batch_size, schedule, seed,
                input_fn=identity, train_eval_idx=None, step_callback=None, callback_steps=(),
                tag="", print_every=10):
    '''AdamW + (cosine | constant) LR schedule, mixed precision on GPU.
    Per epoch: running train loss/acc, eval-mode loss/acc on a fixed training subset, validation loss/acc.
    Keeps the checkpoint with the best validation accuracy (early-stopping selection uses validation only).'''
    model = model.to(DEVICE)
    n = X.shape[0]
    steps_per_epoch = math.ceil(n / batch_size)
    total_steps = epochs * steps_per_epoch
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    if schedule == "cosine":
        lr_lambda = lambda s: 0.5 * (1 + math.cos(math.pi * min(s, total_steps) / total_steps))
    elif schedule == "constant":
        lr_lambda = lambda s: 1.0
    else:
        raise ValueError(schedule)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    gen = torch.Generator().manual_seed(seed)              # data order depends only on the run seed
    callback_steps = set(callback_steps)
    rows, best, step, t0 = [], dict(val_acc=-1.0, epoch=0, state=None), 0, time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(n, generator=gen).to(DEVICE)
        loss_sum = torch.zeros((), device=DEVICE)
        n_correct = torch.zeros((), device=DEVICE)
        for i in range(0, n, batch_size):
            idx = order[i:i + batch_size]
            xb, yb = input_fn(X[idx]), y[idx]
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            loss_sum += loss.detach().float() * len(idx)
            n_correct += (logits.argmax(1) == yb).sum()
            step += 1
            if step_callback is not None and step in callback_steps:
                step_callback(model, step / steps_per_epoch)
                model.train()

        row = dict(epoch=epoch, lr=sched.get_last_lr()[0],
                   train_loss_running=loss_sum.item() / n, train_acc_running=n_correct.item() / n)
        row["val_loss"], row["val_acc"] = loss_acc(predict_logits(model, Xv, input_fn), yv)
        if train_eval_idx is not None:
            row["train_loss"], row["train_acc"] = loss_acc(
                predict_logits(model, X[train_eval_idx], input_fn), y[train_eval_idx])
        rows.append(row)
        if row["val_acc"] > best["val_acc"]:
            best = dict(val_acc=row["val_acc"], epoch=epoch, state=cpu_state(model))
        if print_every and (epoch == 1 or epoch % print_every == 0 or epoch == epochs):
            extra = f" train(eval) {row['train_acc']:.3f}" if "train_acc" in row else ""
            print(f"  [{tag}] ep {epoch:3d}/{epochs} loss {row['train_loss_running']:.3f} "
                  f"train(run) {row['train_acc_running']:.3f}{extra} val {row['val_acc']:.3f} "
                  f"({time.time() - t0:.0f}s)")

    return dict(history=pd.DataFrame(rows), best_epoch=best["epoch"], best_val_acc=best["val_acc"],
                best_state=best["state"], final_state=cpu_state(model), seconds=time.time() - t0,
                steps=step)

## 3. The two input transformations

**(b) Fixed global pixel permutation** (introduced in Q1(a), identical seed and code). One permutation of the 1024
pixel positions is applied to every training, validation and test image, with an RGB pixel moving as a unit. Every pixel value is
still present, but no pixel keeps its neighbours.

**(c) Per-image random tile shuffle, tile size *t*** (new in Q2). Each image is cut into a (32/t)×(32/t) grid of
*t*×*t* tiles, and the tiles are put back in a random order drawn separately for each image. Inside a tile every pixel keeps its
neighbours; across tiles, the arrangement is destroyed and is **different for every image**, so the global layout
carries no usable information. Each image is shuffled **once** (seeded by roll number, tile size and split), so the
training set is a fixed modified dataset rather than augmentation that changes every epoch. Test images are shuffled with the same procedure
and the same tile size.

*Why per-image rather than one fixed tile order?* A single fixed order would still place an object's parts in
consistent (if scrambled) positions, so the global configuration would remain learnable. Part (c) needs global
arrangement to be genuinely **unavailable** while local structure is kept.

The checks below confirm that every condition keeps exactly the same multiset of RGB pixel values per image. They also measure
how much spatial structure survives: the fraction of pixel pairs at distance *d* that are still at distance *d*, in the
same direction, after the transformation.

In [ ]:
# (b) Fixed global pixel permutation: identical to Q1(a)
PIXEL_PERM = torch.from_numpy(
    np.random.default_rng(derive_seed("pixel-permutation")).permutation(32 * 32)).to(DEVICE)

def permute_pixels(x):
    '''Q1(a) transformation: one fixed permutation of the 1024 pixel positions; RGB values move together.'''
    n = x.shape[0]
    return x.reshape(n, 3, 32 * 32)[:, :, PIXEL_PERM].reshape(n, 3, 32, 32)

# (c) Per-image random tile shuffle: new in Q2
def tile_perms(n, tile, split):
    '''One random order of the (32/tile)^2 tile positions per image, seeded by roll number, tile size and split.'''
    G = 32 // tile
    g = torch.Generator().manual_seed(derive_seed("tile-shuffle", tile, split))
    return torch.argsort(torch.rand(n, G * G, generator=g), dim=1)

def tile_shuffle(X, tile, perms, chunk=5000):
    '''Cut every image into a GxG grid of tile x tile squares and reassemble them in that image's own order.
    No pixel value is changed, added or removed; pixels inside a tile keep their relative positions.'''
    G = 32 // tile
    out = torch.empty_like(X)
    perms = perms.to(X.device)
    for i in range(0, X.shape[0], chunk):
        x, p = X[i:i + chunk], perms[i:i + chunk]
        B = x.shape[0]
        tiles = x.reshape(B, 3, G, tile, G, tile).permute(0, 2, 4, 1, 3, 5).reshape(B, G * G, 3, tile, tile)
        tiles = tiles[torch.arange(B, device=X.device)[:, None], p]
        out[i:i + chunk] = tiles.reshape(B, G, G, 3, tile, tile).permute(0, 3, 1, 4, 2, 5).reshape(B, 3, 32, 32)
    return out

CONDITIONS = {"clean": dict(label="clean images", part="(a)"),
              "perm": dict(label="fixed global pixel permutation", part="(b)")}
for t in TILE_SIZES:
    CONDITIONS[f"tile{t}"] = dict(label=f"random {t}x{t}-tile shuffle", part="(c)", tile=t)
TILE_CONDS = [f"tile{t}" for t in TILE_SIZES]
T_KEY = 8 if 8 in TILE_SIZES else TILE_SIZES[len(TILE_SIZES) // 2]   # tile size used for the (c) decision rule
KEY = f"tile{T_KEY}"

def input_fn_for(cond):
    return permute_pixels if cond == "perm" else identity

def materialise(cond, X, split):
    '''Images of one split under a condition. clean/perm return X (the permutation is applied on the fly through
    input_fn, exactly as in Q1); tile conditions return a shuffled copy.'''
    if cond in ("clean", "perm"):
        return X
    t = CONDITIONS[cond]["tile"]
    return tile_shuffle(X, t, tile_perms(X.shape[0], t, split))

def view(cond, X, split):
    '''What the model actually sees.'''
    return input_fn_for(cond)(materialise(cond, X, split))

# ---- Check 1: identical pixel values; Check 2: how much spatial structure survives ----
def pixel_multiset_equal(A, B):
    w = torch.tensor([1.0, 1e3, 1e6], dtype=torch.float64, device=A.device)
    key = lambda Z: (Z.double().reshape(Z.shape[0], 3, -1) * w[None, :, None]).sum(1).sort(dim=1).values
    return torch.equal(key(A), key(B))

def surviving_pairs(cond, d, n=200):
    '''Fraction of pixel pairs at horizontal/vertical distance d that remain at distance d in the same direction.'''
    pos = torch.arange(1024, dtype=torch.float32, device=DEVICE).reshape(1, 1, 32, 32).expand(n, 3, 32, 32).contiguous()
    O = view(cond, pos, "check")[:, 0].long()
    same_row = (O[:, :, d:] // 32) == (O[:, :, :-d] // 32)
    h = ((O[:, :, d:] == O[:, :, :-d] + d) & same_row).float().mean()
    v = (O[:, d:, :] == O[:, :-d, :] + 32 * d).float().mean()
    return float((h + v) / 2)

sample = Xtr[:1000]
rows = []
for cond in CONDITIONS:
    Z = view(cond, sample, "check")
    rows.append(dict(condition=cond, part=CONDITIONS[cond]["part"], description=CONDITIONS[cond]["label"],
                     same_pixel_values=pixel_multiset_equal(sample, Z),
                     positions_unchanged=float((Z == sample).all(dim=1).float().mean()),
                     **{f"pairs_kept_d={d}": surviving_pairs(cond, d) for d in (1, 2, 4, 8, 16)}))
TRANSFORM_TABLE = pd.DataFrame(rows).set_index("condition")
assert TRANSFORM_TABLE["same_pixel_values"].all()
TRANSFORM_TABLE.to_csv(os.path.join(OUT_DIR, "table_q2_transformations.csv"))
display(TRANSFORM_TABLE.round(3))

fig, axes = plt.subplots(len(CONDITIONS), 6, figsize=(8.5, 1.55 * len(CONDITIONS)))
idx = [int(np.where(y_all[train_idx] == c)[0][0]) for c in range(6)]
for i, cond in enumerate(CONDITIONS):
    V = view(cond, Xtr[:max(idx) + 1], "check")
    for j, k in enumerate(idx):
        img = (V[k].permute(1, 2, 0).cpu() * CH_STD + CH_MEAN).clamp(0, 1).numpy()
        ax = axes[i, j]
        ax.imshow(img); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if i == 0:
            ax.set_title(CLASSES[int(y_all[train_idx][k])], fontsize=8)
        if j == 0:
            ax.set_ylabel(cond, fontsize=8)
fig.suptitle("Fig. Q2-0: the same training images under each condition", fontsize=10)
fig.tight_layout()
savefig(fig, "fig_q2_0_conditions.png")

## 4(a). One CNN vs one MLP under a fair comparison

### Fairness criterion (fixed before any result was looked at; first introduced in Q1(a))
A comparison is fair here if the two models differ **only in architecture** and have the same opportunity to learn:

1. **Same number of trainable parameters,** within 0.2%: MLP-M's width (305) is solved to match the CNN's parameter count. The question is whether *how* parameters are arranged matters, not how many there are.
2. **Same data and preprocessing:** the same 45k/5k split and normalisation, and **no data augmentation** for either model. Crops and flips would hand both models a hard-coded spatial prior, which is the very factor under study.
3. **Same training budget:** 30 epochs at batch size 128 (the same number of updates and example presentations), with the same AdamW optimiser, cosine schedule, weight decay, loss, BatchNorm and ReLU.
4. **Same tuning effort:** the same 3-value learning-rate grid for each model, with the LR and the early-stopping epoch chosen **on validation accuracy only**.
5. **Same statistical treatment:** 3 seeds each (shared seed streams, so the comparison is paired). The final comparison is mean ± sd **test** accuracy, evaluated once.
6. **Declared imbalance:** compute per image is **not** matched. Weight sharing makes the CNN perform about 118× more multiply-accumulates per image. Part (b) controls for this, because permuted inputs leave the FLOPs unchanged.

**Primary outcome:** test accuracy of the best-validation checkpoint (mean ± sd over seeds). The training-set fit
(eval-mode accuracy on a fixed 5k training subset) is recorded to tell an underfitting model apart from one that generalises poorly.

In [ ]:
CNN_PARAMS = count_params(CNN())
W_MATCH = width_matching(CNN_PARAMS)
ARCHS = {"MLP-M": dict(kind="mlp", width=W_MATCH), "CNN": dict(kind="cnn")}

def build_model(cfg):
    return MLP(cfg["width"]) if cfg["kind"] == "mlp" else CNN()

rows = []
for name, cfg in ARCHS.items():
    m = build_model(cfg)
    layers = [(n, p.numel()) for n, p in m.named_parameters() if p.dim() > 1]
    rows.append(dict(model=name, architecture=m.spec, parameters=count_params(m),
                     weight_tensors=" + ".join(f"{k:,}" for _, k in layers),
                     MACs_per_image=count_macs(m), introduced_in="Q1(a)"))
MODEL_TABLE = pd.DataFrame(rows).set_index("model")
MODEL_TABLE["params_vs_CNN"] = (MODEL_TABLE["parameters"] / CNN_PARAMS).round(4)
MODEL_TABLE.to_csv(os.path.join(OUT_DIR, "table_q2_models.csv"))
print("Table Q2-1: compared models (both reused from Q1(a))")
with pd.option_context("display.max_colwidth", 120):
    display(MODEL_TABLE)
print(f"Training budget for every run: {CFG_A['epochs']} epochs x {math.ceil(N_TRAIN / CFG_A['batch_size'])} steps, "
      f"batch {CFG_A['batch_size']}, AdamW + cosine, wd {CFG_A['weight_decay']}, LR grid {LR_GRID}, {N_SEEDS} seeds")

In [ ]:
# ---- Run machinery: reuses Q1(a)'s cache for configurations that Q1 already trained ----
CACHE_TAG = hashlib.md5(json.dumps([SEED, N_TRAIN, N_VAL, CFG_A, TRAIN_EVAL_SUBSET],
                                   sort_keys=True).encode()).hexdigest()[:8]     # identical to Q1's cache tag
TRAIN_EVAL_IDX = torch.from_numpy(np.random.default_rng(derive_seed("train-eval-subset")).choice(
    N_TRAIN, TRAIN_EVAL_SUBSET, replace=False)).to(DEVICE)                       # identical to Q1
Q1_NAMES = {("MLP-M", "clean"): "MLP-M", ("CNN", "clean"): "CNN",
            ("MLP-M", "perm"): "MLP-M-perm", ("CNN", "perm"): "CNN-perm"}
SOURCE, RESULTS = {}, {}

def run(arch, cond, lr, seed_idx, Xtr_c, Xva_c):
    from_q1 = (arch, cond) in Q1_NAMES
    path = (os.path.join(Q1_RUNS, f"q1a_{CACHE_TAG}_{Q1_NAMES[(arch, cond)]}_lr{lr:g}_s{seed_idx}.pt") if from_q1
            else os.path.join(OUT_DIR, "runs", f"q2_{CACHE_TAG}_{arch}_{cond}_lr{lr:g}_s{seed_idx}.pt"))
    if os.path.exists(path):
        out = torch.load(path, map_location="cpu")
        src = "reused: loaded from Q1(a) run cache" if from_q1 else "loaded from Q2 cache"
    else:
        seed = derive_seed("q1a-run", seed_idx)          # same seed stream as Q1 -> paired across conditions
        torch.manual_seed(seed)
        model = build_model(ARCHS[arch])
        res = train_model(model, Xtr_c, ytr, Xva_c, yva, lr=lr, seed=seed, input_fn=input_fn_for(cond),
                          train_eval_idx=TRAIN_EVAL_IDX, tag=f"{arch}/{cond} lr={lr:g} seed#{seed_idx}", **CFG_A)
        out = dict(name=Q1_NAMES.get((arch, cond), f"{arch}-{cond}"), lr=lr, seed_idx=seed_idx,
                   history=df_to_lists(res["history"]), best_epoch=int(res["best_epoch"]),
                   best_val_acc=float(res["best_val_acc"]), best_state=res["best_state"],
                   seconds=float(res["seconds"]))
        torch.save(out, path)
        src = ("retrained here with the Q1(a) configuration (Q1 cache absent)" if from_q1
               else "trained here (new in Q2)")
    out["history"] = pd.DataFrame(out["history"])
    SOURCE[(arch, cond, lr, seed_idx)] = src
    return out

def fit_condition(arch, cond, Xtr_c, Xva_c, tune=True):
    '''LR chosen on validation (seed #0) from the shared grid, then repeated with further seeds at that LR.'''
    grid = LR_GRID if tune else [RESULTS[(arch, "clean")]["lr"]]
    sweep = {lr: run(arch, cond, lr, 0, Xtr_c, Xva_c) for lr in grid}
    lr = max(grid, key=lambda l: sweep[l]["best_val_acc"])
    runs = [sweep[lr]] + [run(arch, cond, lr, s, Xtr_c, Xva_c) for s in range(1, N_SEEDS)]
    for l, r in sweep.items():
        if l != lr:
            r["best_state"] = None
    RESULTS[(arch, cond)] = dict(lr=lr, runs=runs, sweep={l: r["best_val_acc"] for l, r in sweep.items()})
    return RESULTS[(arch, cond)]

def val_table(conds):
    rows = []
    for cond in conds:
        for arch in ARCHS:
            R = RESULTS[(arch, cond)]
            va = [r["best_val_acc"] for r in R["runs"]]
            rows.append(dict(condition=cond, model=arch, selected_lr=R["lr"], val_acc=mean_sd(va, 4),
                             val_acc_mean=np.mean(va), best_epoch=np.mean([r["best_epoch"] for r in R["runs"]]),
                             train_acc_final=np.mean([r["history"]["train_acc"].iloc[-1] for r in R["runs"]]),
                             **{f"val@lr={l:g}": round(v, 4) for l, v in R["sweep"].items()}))
    return pd.DataFrame(rows).set_index(["condition", "model"])

def plot_curves(pairs, title, fname):
    fig, axes = plt.subplots(1, len(pairs), figsize=(3.9 * len(pairs), 3.4), squeeze=False)
    for j, (arch, cond) in enumerate(pairs):
        ax, runs = axes[0, j], RESULTS[(arch, cond)]["runs"]
        ep = runs[0]["history"]["epoch"].values
        for col, lab, c in [("train_acc", "train (eval mode)", PALETTE[0]), ("val_acc", "validation", PALETTE[1])]:
            M = np.stack([r["history"][col].values for r in runs])
            ax.plot(ep, M.mean(0), color=c, label=lab)
            ax.fill_between(ep, M.mean(0) - M.std(0), M.mean(0) + M.std(0), color=c, alpha=0.2, linewidth=0)
        ax.set_ylim(0, 1.02)
        ax.set_xlabel("epoch")
        ax.set_title(f"{arch}, {cond} (LR {RESULTS[(arch, cond)]['lr']:g})", fontsize=9)
    axes[0, 0].set_ylabel("accuracy")
    axes[0, 0].legend(fontsize=8, loc="lower right")
    fig.suptitle(f"{title} (mean ± sd over {N_SEEDS} seeds)", fontsize=10)
    fig.tight_layout()
    savefig(fig, fname)

T0 = time.time()
for arch in ARCHS:
    fit_condition(arch, "clean", Xtr, Xva)
VAL_A = val_table(["clean"])
print("Validation results, clean images (selection only; final numbers are test results in Section 5)")
display(VAL_A.drop(columns="val_acc_mean"))
plot_curves([("MLP-M", "clean"), ("CNN", "clean")], "Fig. Q2-1: training and validation curves, clean images",
            "fig_q2_1_curves_clean.png")

### Looking inside the two models: first-layer weights
Each first-layer unit of the MLP is a weight vector over all 3072 inputs, so it can be shown as a 32×32 RGB "template".
The CNN's first layer is 32 filters of size 3×3×3, applied at every position.

In [ ]:
def norm01(a):
    a = a - a.min()
    return a / max(float(a.max()), 1e-12)

mlp_state = RESULTS[("MLP-M", "clean")]["runs"][0]["best_state"]
cnn_state = RESULTS[("CNN", "clean")]["runs"][0]["best_state"]
W = mlp_state["net.1.weight"].float().reshape(-1, 3, 32, 32)
top = W.flatten(1).norm(dim=1).argsort(descending=True)[:12]
K = cnn_state["features.0.weight"].float()

fig = plt.figure(figsize=(12, 3.6))
sub_l, sub_r = fig.subfigures(1, 2, width_ratios=[1, 1.1])
ax_l = sub_l.subplots(3, 4)
for ax, u in zip(ax_l.ravel(), top):
    ax.imshow(norm01(W[u]).permute(1, 2, 0).numpy()); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
sub_l.suptitle("MLP-M: 12 largest-norm first-layer units\n(each spans the whole 32x32 image)", fontsize=9)
ax_r = sub_r.subplots(4, 8)
for ax, k in zip(ax_r.ravel(), K):
    ax.imshow(norm01(k).permute(1, 2, 0).numpy(), interpolation="nearest"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
sub_r.suptitle("CNN: all 32 first-layer 3x3 filters\n(each reused at every position)", fontsize=9)
savefig(fig, "fig_q2_2_first_layer_weights.png")

R_m, R_c = RESULTS[("MLP-M", "clean")], RESULTS[("CNN", "clean")]
fit_m = np.mean([r["history"]["train_acc"].iloc[-1] for r in R_m["runs"]])
fit_c = np.mean([r["history"]["train_acc"].iloc[-1] for r in R_c["runs"]])
va_m = np.mean([r["best_val_acc"] for r in R_m["runs"]])
va_c = np.mean([r["best_val_acc"] for r in R_c["runs"]])
print(f"Validation: CNN {pct(va_c)} vs MLP-M {pct(va_m)} ({pp(va_c - va_m)}).")
print(f"Training-set fit at the final epoch (eval mode): MLP-M {pct(fit_m)}, CNN {pct(fit_c)}; "
      f"train-validation gap MLP-M {pp(fit_m - va_m)}, CNN {pp(fit_c - va_c)}.")

### Proposed reason for the difference (stated before parts (b) and (c) are run)
At equal parameters and an equal budget, the gap does not come from the MLP being unable to fit: the curves above show
the MLP fitting its training set while generalising worse (a large train–validation gap). Its first-layer units are
whole-image, position-specific templates, whereas the CNN's are small local detectors applied everywhere.

**Proposed reason:** *the CNN generalises better because locality and weight sharing let it exploit the strong correlations
between neighbouring pixels. It learns a small set of local pattern detectors (edges, colour blobs, textures) that it
reuses at every position. The MLP has to learn each pattern separately at each position, so with 45k images it ends up fitting
position-specific pixel combinations that do not transfer.*

This makes a sharp, testable prediction for part (b).

## 4(b). Same pixel values, spatial structure destroyed

**Modification.** This is the fixed global pixel permutation (Section 3), first introduced in Q1(a). Every pixel value stays in the
image (checked above), but no pixel keeps its neighbours (pairs kept ≈ 0 at every distance). The same permutation is
applied to the training, validation and test images, and both models are **retrained** on the permuted data with the identical
budget and LR grid. These runs were first trained in Q1(a) and are loaded from its cache.

### Expected result (stated beforehand)
| | Expected change in test accuracy | Why |
|---|---|---|
| **MLP-M** | **none**, within seed noise (\|Δ\| ≤ 2 seed sd, at least 1 pp) | a fixed permutation of the inputs is absorbed by permuting the first layer's columns, and the i.i.d. initialisation does not care about input order |
| **CNN** | **large drop, to about the MLP's level or below** (CNN–MLP gap shrinks by more than half) | the 3×3 filters now see unrelated pixels, so the local correlations they rely on are gone |

**How this tests the reason from (a).** Permutation leaves the parameter count, FLOPs, optimiser, budget and pixel values
unchanged and removes only neighbourhood structure. If the CNN kept its advantage, the advantage would have to come
from something other than local spatial correlations, such as extra compute or easier optimisation, and the reason from (a) would be wrong.

In [ ]:
for arch in ARCHS:
    fit_condition(arch, "perm", Xtr, Xva)
VAL_B = val_table(["clean", "perm"])
print("Validation results, clean vs permuted (selection only)")
display(VAL_B.drop(columns="val_acc_mean"))
plot_curves([("MLP-M", "perm"), ("CNN", "perm")], "Fig. Q2-3: curves on permuted images", "fig_q2_3_curves_perm.png")

## 4(c). Local structure or global arrangement?

Part (b) shows *whether* the CNN needs spatial structure, but a full permutation destroys every kind of structure at
once. It cannot separate two explanations for **what** the CNN uses:

* **E-local (bag of local features):** the CNN's advantage comes from **local** structure, meaning patterns within a few pixels
  (edges, textures, colour blobs), pooled over the image. Its global average pooling makes it close to a "bag of
  local features": where those patterns sit relative to each other matters little.
* **E-global (shape and configuration):** the CNN's advantage comes from the **global arrangement** of parts, meaning object shape
  and layout across the image, built up through its deeper, larger receptive fields.

**Experiment.** Train and test both models on **per-image random tile shuffles** with *t* = 16, 8 and 4 px (Section 3).
Shuffling keeps all structure *inside* a *t*×*t* tile and makes the global arrangement random, so it carries no information.
Each condition gets the same treatment as in (a): identical budget, LR chosen from the same grid on validation
(`TUNE_LR_C`), and 3 seeds.

### Predictions (stated beforehand)
| Measurement | E-local | E-global |
|---|---|---|
| CNN accuracy with 8 px tiles | stays close to clean | falls towards the permuted-image level |
| **CNN retention R(t)** = (acc(t) − acc(perm)) / (acc(clean) − acc(perm)) | R(8) ≥ 0.5 | R(8) < 0.5, and already low at t = 16 |
| CNN accuracy as *t* shrinks to 4 | falls only once tiles get close to the size of its local features | flat and low |
| CNN–MLP gap under tile shuffling | stays large | shrinks towards 0 |
| CNN confidence on its **correct** predictions | stays near the clean level (decisions still rest on strong local evidence) | falls towards the permuted-image level (weak, colour-statistics evidence) |

The reason from (a) gives one more prediction, the same under both explanations. **The MLP should get worse with tile shuffling**
(unlike with the fixed permutation in (b)), because its position-specific templates need content to appear in
consistent positions, and per-image shuffling removes that.

**Decision rule (fixed in advance).** E-local is supported if R(8) ≥ 0.5 on the test set, and E-global if R(8) < 0.5. As a
secondary check on confidence: Rc(8), the same normalisation applied to the CNN's mean confidence on its correct
predictions, should agree.

**Error and confidence analysis (test set, all conditions):** accuracy, NLL, mean confidence, confidence on correct vs
wrong predictions, expected calibration error (ECE, 15 bins), AUROC of confidence for telling right from wrong,
reliability diagrams, per-class accuracy and the CNN's confusion matrices.

In [ ]:
for cond in TILE_CONDS:
    Xtr_c, Xva_c = materialise(cond, Xtr, "train"), materialise(cond, Xva, "val")
    for arch in ARCHS:
        fit_condition(arch, cond, Xtr_c, Xva_c, tune=TUNE_LR_C)
    del Xtr_c, Xva_c
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
VAL_ALL = val_table(list(CONDITIONS))
VAL_ALL.to_csv(os.path.join(OUT_DIR, "table_q2_validation.csv"))
print(f"Validation results, all conditions (selection only). Q2 wall-clock this session: {(time.time() - T0) / 60:.1f} min")
display(VAL_ALL.drop(columns="val_acc_mean"))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, arch in zip(axes, ARCHS):
    for k, cond in enumerate(CONDITIONS):
        M = np.stack([r["history"]["val_acc"].values for r in RESULTS[(arch, cond)]["runs"]])
        ax.plot(np.arange(1, M.shape[1] + 1), M.mean(0), color=PALETTE[k], label=cond)
    ax.set_title(arch, fontsize=10)
    ax.set_xlabel("epoch")
axes[0].set_ylabel("validation accuracy")
axes[1].legend(fontsize=8, loc="lower right")
fig.suptitle("Fig. Q2-4: validation curves for every condition (mean over seeds)", fontsize=10)
fig.tight_layout()
savefig(fig, "fig_q2_4_val_curves_all.png")

## 5. Final test-set evaluation (once, after every choice is fixed)
The official test set is loaded here for the first time. Each test image is transformed exactly as the training images of
the same condition were: no change for clean, the same fixed permutation for (b), and the same tile size and per-image
shuffling procedure for (c). Every model is its best-validation checkpoint.

In [ ]:
test_raw = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True)
Xte = to_tensor(np.asarray(test_raw.data)).to(DEVICE)
yte = torch.from_numpy(np.asarray(test_raw.targets, dtype=np.int64))

TEST_LOGITS = {}
for cond in CONDITIONS:
    Xte_c = materialise(cond, Xte, "test")
    for arch in ARCHS:
        TEST_LOGITS[(arch, cond)] = []
        for r in RESULTS[(arch, cond)]["runs"]:
            model = build_model(ARCHS[arch]).to(DEVICE)
            model.load_state_dict(r["best_state"])
            TEST_LOGITS[(arch, cond)].append(predict_logits(model, Xte_c, input_fn_for(cond)).cpu())
    del Xte_c

ACC = {k: np.array([(lg.argmax(1) == yte).float().mean().item() for lg in v]) for k, v in TEST_LOGITS.items()}
rows = []
for cond in CONDITIONS:
    gap = ACC[("CNN", cond)] - ACC[("MLP-M", cond)]
    rows.append({"part": CONDITIONS[cond]["part"], "condition": cond, "description": CONDITIONS[cond]["label"],
                 f"MLP-M test acc ({count_params(MLP(W_MATCH)):,} params)": mean_sd(ACC[("MLP-M", cond)], 4),
                 f"CNN test acc ({CNN_PARAMS:,} params)": mean_sd(ACC[("CNN", cond)], 4),
                 "CNN - MLP (paired)": mean_sd(gap, 4)})
TEST_TABLE = pd.DataFrame(rows).set_index("condition")
TEST_TABLE.to_csv(os.path.join(OUT_DIR, "table_q2_test_results.csv"))
print(f"Table Q2-2: TEST accuracy (mean ± sd over {N_SEEDS} seeds, best-validation checkpoints)")
with pd.option_context("display.max_colwidth", 60):
    display(TEST_TABLE)

m = lambda a, c: float(ACC[(a, c)].mean())
SEED_SD = float(np.sqrt(np.nanmean([np.var(v, ddof=1) for v in ACC.values()]))) if N_SEEDS > 1 else float("nan")
def retention(num, den, min_den=0.01):
    '''Share of the clean-vs-permuted difference that is retained; undefined (nan) if there is no difference to retain.'''
    return num / den if den > min_den else float("nan")

R = {c: retention(m("CNN", c) - m("CNN", "perm"), m("CNN", "clean") - m("CNN", "perm")) for c in TILE_CONDS}

fig, ax = plt.subplots(figsize=(7, 4))
xs = ["clean\n(t = 32)"] + [f"t = {t}" for t in TILE_SIZES]
for k, arch in enumerate(ARCHS):
    vals = [ACC[(arch, "clean")]] + [ACC[(arch, c)] for c in TILE_CONDS]
    ax.errorbar(range(len(xs)), [v.mean() for v in vals], yerr=[np.std(v) for v in vals], marker="o", ms=7,
                capsize=3, color=PALETTE[k], label=f"{arch}: tile shuffle")
    ax.axhline(m(arch, "perm"), color=PALETTE[k], ls="--", lw=1.2, alpha=0.8)
    ax.text(len(xs) - 1 + 0.08, m(arch, "perm"), f"{arch}, (b) permutation", color=PALETTE[k], fontsize=7,
            va="bottom" if arch == "CNN" else "top")
for i, c in enumerate(TILE_CONDS, start=1):
    ax.annotate(f"R = {R[c]:.2f}", (i, m("CNN", c)), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8)
ax.set_xticks(range(len(xs)))
ax.set_xticklabels(xs)
ax.set_xlim(-0.3, len(xs) + 0.9)
ax.set_ylim(0, 1)
ax.set_xlabel("tile size t (local structure kept up to t px; global arrangement random)")
ax.set_ylabel("test accuracy")
ax.set_title("Fig. Q2-5: test accuracy as global arrangement is removed\n"
             "(R = share of the CNN's spatial benefit retained)", fontsize=10)
ax.legend(fontsize=8, loc="lower left")
savefig(fig, "fig_q2_5_tile_curve.png")

### Error and confidence analysis (test set)

In [ ]:
def confidence_metrics(logits, y, n_bins=15):
    p = logits.softmax(1)
    conf, pred = p.max(1)
    correct = pred == y
    acc = correct.float().mean().item()
    edges = torch.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        b = (conf > lo) & (conf <= hi)
        if b.any():
            ece += b.float().mean().item() * abs(correct[b].float().mean().item() - conf[b].mean().item())
    c = correct.numpy()
    return dict(acc=acc, nll=F.cross_entropy(logits, y).item(), mean_conf=conf.mean().item(),
                conf_correct=conf[correct].mean().item() if correct.any() else float("nan"),
                conf_wrong=conf[~correct].mean().item() if (~correct).any() else float("nan"),
                overconfidence=conf.mean().item() - acc, ece=ece,
                auroc_conf_vs_correct=roc_auc_score(c, conf.numpy()) if 0 < c.mean() < 1 else float("nan"))

rows, METRICS = [], {}
for cond in CONDITIONS:
    for arch in ARCHS:
        per_seed = pd.DataFrame([confidence_metrics(lg, yte) for lg in TEST_LOGITS[(arch, cond)]])
        METRICS[(arch, cond)] = per_seed.mean()
        rows.append(dict(condition=cond, model=arch, **{k: f"{per_seed[k].mean():.3f} ± {per_seed[k].std():.3f}"
                                                        if N_SEEDS > 1 else f"{per_seed[k].mean():.3f}"
                                                        for k in per_seed.columns}))
CONF_TABLE = pd.DataFrame(rows).set_index(["condition", "model"])
CONF_TABLE.to_csv(os.path.join(OUT_DIR, "table_q2_confidence.csv"))
print("Table Q2-3: test-set accuracy, calibration and confidence (mean ± sd over seeds)")
display(CONF_TABLE)

# ---- Fig. Q2-6: reliability diagrams (predictions pooled over seeds) ----
def pooled(arch, cond):
    lg = torch.cat(TEST_LOGITS[(arch, cond)])
    p = lg.softmax(1)
    conf, pred = p.max(1)
    return conf.numpy(), (pred == yte.repeat(len(TEST_LOGITS[(arch, cond)]))).numpy(), pred.numpy()

fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(3.1 * len(CONDITIONS), 3.3), sharey=True)
edges = np.linspace(0, 1, 16)
for ax, cond in zip(axes, CONDITIONS):
    ax.plot([0, 1], [0, 1], color=GREY, lw=1, ls=":")
    for k, arch in enumerate(ARCHS):
        conf, corr, _ = pooled(arch, cond)
        b = np.clip(np.digitize(conf, edges) - 1, 0, 14)
        xs_, ys_ = [], []
        for i in range(15):
            if (b == i).sum() >= 30:
                xs_.append(conf[b == i].mean()); ys_.append(corr[b == i].mean())
        ax.plot(xs_, ys_, marker="o", ms=4, color=PALETTE[k], label=arch)
    ax.set_title(cond, fontsize=9)
    ax.set_xlabel("confidence")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
axes[0].set_ylabel("accuracy")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle("Fig. Q2-6: reliability diagrams on the test set (dotted = perfect calibration)", fontsize=10)
fig.tight_layout()
savefig(fig, "fig_q2_6_reliability.png")

# ---- Fig. Q2-7: confidence of correct vs wrong predictions ----
show_conds = ["clean", KEY, "perm"]
fig, axes = plt.subplots(2, 3, figsize=(11, 5.4), sharex=True)
bins = np.linspace(0, 1, 26)
for i, arch in enumerate(ARCHS):
    for j, cond in enumerate(show_conds):
        conf, corr, _ = pooled(arch, cond)
        ax = axes[i, j]
        if corr.any():
            ax.hist(conf[corr], bins=bins, density=True, alpha=0.6, color=PALETTE[0], label="correct")
        if (~corr).any():
            ax.hist(conf[~corr], bins=bins, density=True, alpha=0.6, color=PALETTE[1], label="wrong")
        ax.set_title(f"{arch}, {cond}: acc {corr.mean():.3f}, AUROC "
                     f"{METRICS[(arch, cond)]['auroc_conf_vs_correct']:.3f}", fontsize=8)
        if j == 0:
            ax.set_ylabel("density")
        if i == 1:
            ax.set_xlabel("max softmax confidence")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Fig. Q2-7: test-set confidence of correct and wrong predictions", fontsize=10)
fig.tight_layout()
savefig(fig, "fig_q2_7_confidence_hist.png")

# ---- Fig. Q2-8: per-class test accuracy ----
y_np = yte.numpy()
pc_rows, pc_index = [], []
for arch in ARCHS:
    for cond in CONDITIONS:
        preds = np.stack([lg.argmax(1).numpy() for lg in TEST_LOGITS[(arch, cond)]])
        pc_rows.append([(preds[:, y_np == c] == c).mean() for c in range(10)])
        pc_index.append(f"{arch} | {cond}")
PER_CLASS = pd.DataFrame(pc_rows, index=pc_index, columns=CLASSES)
PER_CLASS.to_csv(os.path.join(OUT_DIR, "table_q2_per_class_accuracy.csv"))
fig, ax = plt.subplots(figsize=(10.5, 0.45 * len(PER_CLASS) + 1.2))
im = ax.imshow(PER_CLASS.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
for (i, j), v in np.ndenumerate(PER_CLASS.values):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if v > 0.6 else "#0b0b0b")
ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=30, ha="right")
ax.set_yticks(range(len(PER_CLASS))); ax.set_yticklabels(PER_CLASS.index, fontsize=8)
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.025, label="test accuracy")
ax.set_title("Fig. Q2-8: per-class test accuracy (pooled over seeds)", fontsize=10)
savefig(fig, "fig_q2_8_per_class.png")

drop = (PER_CLASS.loc[f"CNN | clean"] - PER_CLASS.loc[f"CNN | {KEY}"]).sort_values(ascending=False)
print(f"CNN per-class accuracy lost going from clean to {KEY} (largest first):")
display(drop.round(3).to_frame("accuracy lost").T)

# ---- Fig. Q2-9: CNN confusion matrices ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, cond in zip(axes, ["clean", KEY]):
    _, _, pred = pooled("CNN", cond)
    yy = np.tile(y_np, len(TEST_LOGITS[("CNN", cond)]))
    cm = np.zeros((10, 10))
    np.add.at(cm, (yy, pred), 1)
    cm = cm / cm.sum(1, keepdims=True)
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for (i, j), v in np.ndenumerate(cm):
        if v >= 0.05:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if v > 0.6 else "#0b0b0b")
    ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES, fontsize=7)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.grid(False)
    ax.set_title(f"CNN, {cond}", fontsize=9)
fig.suptitle("Fig. Q2-9: CNN test confusion matrices (rows normalised; entries < 0.05 hidden)", fontsize=10)
fig.tight_layout()
savefig(fig, "fig_q2_9_confusion_cnn.png")

## 6. Results of the three tests and conclusion
The next cell applies the predictions and decision rules written above to the **test-set** numbers and writes the
conclusion. The text is also saved to `q2_outputs/conclusion.txt`.

In [ ]:
band = max(2 * SEED_SD, 0.01) if not np.isnan(SEED_SD) else 0.01
gap = {c: m("CNN", c) - m("MLP-M", c) for c in CONDITIONS}
mc = lambda a, c, k: float(METRICS[(a, c)][k])
TXT = {}

# ---------- (a) ----------
TXT["a"] = (
    f"(a) Under the fairness criterion fixed in advance (equal parameters: MLP-M {count_params(MLP(W_MATCH)):,} vs CNN "
    f"{CNN_PARAMS:,}; same data, no augmentation, same optimiser, schedule, epochs and batch size; same LR grid chosen on "
    f"validation; {N_SEEDS} paired seeds), the CNN reaches {pct(m('CNN', 'clean'))} test accuracy against "
    f"{pct(m('MLP-M', 'clean'))} for the MLP ({pp(gap['clean'])}). The MLP fits {pct(fit_m)} of its training subset "
    f"(CNN {pct(fit_c)}), " +
    ("so the gap is one of generalisation, not of fitting. " if (fit_m >= 0.9 and gap["clean"] > 0) else
     "so this run does not show the expected pattern of a well-fitting but worse-generalising MLP; treat the proposed "
     "reason as a hypothesis that the evidence here does not yet support. ") +
    f"Its first-layer units are whole-image templates "
    f"while the CNN's are 3x3 detectors reused everywhere. Proposed reason: locality and weight sharing let the CNN exploit "
    f"the correlations between neighbouring pixels with reusable local features, which the MLP has to relearn at every "
    f"position.")

# ---------- (b) ----------
dm, dc = m("MLP-M", "perm") - m("MLP-M", "clean"), m("CNN", "perm") - m("CNN", "clean")
mlp_inv = abs(dm) <= band
cnn_coll = gap["perm"] <= 0.5 * gap["clean"] and dc < -band
TXT["b"] = (
    f"(b) Expected beforehand: no change for the MLP, and a large CNN drop to about the MLP's level. Observed on the test set: "
    f"MLP-M {pp(dm)} ({'within' if mlp_inv else 'outside'} the noise band of ±{100 * band:.1f} pp), CNN {pp(dc)}; the "
    f"CNN-MLP gap goes from {pp(gap['clean'])} to {pp(gap['perm'])}. " +
    ("This is the predicted pattern. With parameters, FLOPs, budget and the pixel values themselves unchanged, the CNN's "
     "advantage disappears once neighbourhoods are destroyed, so it depends on local spatial structure. This supports "
     "the reason from (a) and rules out extra compute or easier optimisation as the explanation."
     if (mlp_inv and cnn_coll) else
     "This departs from the prediction (" +
     ", ".join(s for s, ok in [("the MLP changed", not mlp_inv), ("the CNN kept much of its advantage", not cnn_coll)] if ok)
     + "), so the reason from (a) is at best incomplete."))

# ---------- (c) ----------
Rc = {c: retention(mc("CNN", c, "conf_correct") - mc("CNN", "perm", "conf_correct"),
                   mc("CNN", "clean", "conf_correct") - mc("CNN", "perm", "conf_correct")) for c in TILE_CONDS}
decidable = not np.isnan(R[KEY])
local = decidable and R[KEY] >= 0.5
tile_str = ", ".join(f"t={CONDITIONS[c]['tile']}: CNN {pct(m('CNN', c))} (R={R[c]:.2f}), MLP {pct(m('MLP-M', c))}"
                     for c in TILE_CONDS)
mlp_drop_key = m("MLP-M", KEY) - m("MLP-M", "clean")
TXT["c"] = (
    f"(c) E-local (local features) vs E-global (global arrangement). With per-image random tile shuffling, applied in the same way to "
    f"training and test images: {tile_str}; clean CNN {pct(m('CNN', 'clean'))}, permuted CNN {pct(m('CNN', 'perm'))}. "
    f"At t={T_KEY} the CNN retains R = {R[KEY]:.2f} of its spatial benefit, and the CNN-MLP gap is {pp(gap[KEY])} "
    f"(clean {pp(gap['clean'])}). "
    f"Confidence: the CNN's correct predictions have mean confidence {mc('CNN', 'clean', 'conf_correct'):.3f} (clean), "
    f"{mc('CNN', KEY, 'conf_correct'):.3f} ({KEY}) and {mc('CNN', 'perm', 'conf_correct'):.3f} (perm), so Rc = "
    f"{Rc[KEY]:.2f}. The AUROC of confidence for separating right from wrong is "
    f"{mc('CNN', 'clean', 'auroc_conf_vs_correct'):.3f} / {mc('CNN', KEY, 'auroc_conf_vs_correct'):.3f} / "
    f"{mc('CNN', 'perm', 'auroc_conf_vs_correct'):.3f}, and the ECE is {mc('CNN', 'clean', 'ece'):.3f} / {mc('CNN', KEY, 'ece'):.3f} / "
    f"{mc('CNN', 'perm', 'ece'):.3f}. The classes that lose most for the CNN at t={T_KEY} are "
    f"{', '.join(drop.index[:3])}. "
    f"The MLP changes by {pp(mlp_drop_key)} at t={T_KEY} although it was unaffected by the fixed permutation: its "
    f"features depend on content appearing at consistent absolute positions. "
    + ("Decision: undetermined. The CNN is not better on clean than on permuted images, so there is no spatial "
       "benefit whose retention could be measured." if not decidable else
       f"Decision (pre-registered rule R({T_KEY}) >= 0.5): E-LOCAL is supported. The CNN keeps most of its advantage "
       f"when only structure within {T_KEY}-px tiles survives, so it behaves largely as a bag of local features rather "
       f"than relying on the global shape of the object."
       if local else
       f"Decision (pre-registered rule R({T_KEY}) >= 0.5 not met): E-GLOBAL is supported. Randomising the arrangement "
       f"of {T_KEY}-px tiles removes most of the CNN's advantage, so it depends on how local parts are arranged across the image.")
    + ("" if not decidable else " The confidence check is undefined (no confidence difference between clean and "
       "permuted)." if np.isnan(Rc[KEY]) else " The confidence check agrees." if (Rc[KEY] >= 0.5) == local else
       " The confidence check disagrees with the accuracy-based decision, so treat the decision as tentative."))

# ---------- link ----------
TXT["link"] = (
    "Conclusion linking the three experiments: (a) " +
    ("at equal parameters and budget the CNN generalises better although both models fit their training data, which "
     "points to an inductive-bias explanation; " if (fit_m >= 0.9 and gap["clean"] > 0) else
     "does not show the expected pattern (a CNN advantage with a well-fitting MLP), so the proposed reason is only a "
     "hypothesis; ") +
    "(b) destroying neighbourhoods while "
    "keeping every pixel value " + ("removes the CNN's advantage and leaves the MLP unchanged, " if (mlp_inv and cnn_coll)
                                    else "does not behave as predicted, ") +
    "which tests that explanation directly; (c) " +
    ("cannot separate local from global structure, because the CNN shows no spatial benefit to begin with."
     if not decidable else
     f"shows that the structure the CNN needs is local (within about {T_KEY} px), not the global layout. Randomising the layout "
     "costs the CNN little but hurts the MLP, whose position-specific templates are exactly what (a) proposed as its weakness."
     if local else
     f"shows that local structure alone is not enough: the CNN also relies on how {T_KEY}-px parts are arranged, "
     "so its advantage combines local feature reuse with sensitivity to global configuration."))

CONCLUSION = "\n\n".join(TXT[k] for k in ["a", "b", "c", "link"])
with open(os.path.join(OUT_DIR, "conclusion.txt"), "w") as f:
    f.write(CONCLUSION)
print(CONCLUSION)

### Limitations to state in the report
* **The conclusions are about this CNN.** It ends in global average pooling, which already discards absolute position. A network ending in a large fully connected layer might rely more on global arrangement.
* **Tile boundaries add artificial edges.** Shuffling creates discontinuities at tile borders, so part of the drop at small *t* may come from these new edges rather than from the loss of arrangement.
* **No augmentation and 30 epochs**, so absolute accuracies are below standard CIFAR-10 numbers. That is by design, to keep the comparison controlled.
* **Three seeds.** Differences are reported as mean ± sd, with a noise band of 2 seed sd (at least 1 pp).
* **Learning rate:** with `TUNE_LR_C = True`, every condition gets the same LR grid. The LR is selected on seed #0, which is also one of the three reported seeds, a small optimistic bias that applies equally to every condition.

## 7. Reuse log

| Item | First introduced in | Used in Q2 | Notes |
|---|---|---|---|
| Stratified 45k/5k split (roll-number seed), normalisation statistics | Q1 §1 | all parts | verified identical to Q1's `split_indices.npz` when available |
| `MLP`, `CNN`, `count_params`, `count_macs`, `width_matching` | Q1 §2 | all parts | code copied verbatim |
| `train_model`, `predict_logits`, `df_to_lists` | Q1 §2 | all parts | code copied verbatim |
| **MLP-M** (width 305) and **CNN** (1.13M params), fairness criterion | Q1(a) | Q2(a), (b), (c) | the same two models are compared in every part |
| Trained runs MLP-M / CNN on clean images (LR sweep + seeds) | Q1(a) | **Q2(a) results** | loaded from Q1's run cache, or retrained with identical settings if absent (see the source table below) |
| Fixed pixel permutation and the runs MLP-M-perm / CNN-perm | Q1(a) | **Q2(b) results** | same as above |
| Per-image random tile shuffle; all tile-condition runs; confidence and error analysis | **new in Q2** | Q2(c) | |

In [ ]:
src = pd.Series({f"{a} | {c} | lr={lr:g} | seed#{s}": v for (a, c, lr, s), v in SOURCE.items()}, name="source")
print("Where each run came from:")
display(src.value_counts().to_frame("runs"))
src.to_csv(os.path.join(OUT_DIR, "table_q2_run_sources.csv"))

def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    return o

summary = dict(roll_number=ROLL_NUMBER, seed=SEED, tile_sizes=TILE_SIZES, tune_lr_c=TUNE_LR_C,
               selected_lr={f"{a}|{c}": R_["lr"] for (a, c), R_ in RESULTS.items()},
               test_acc_per_seed={f"{a}|{c}": v.tolist() for (a, c), v in ACC.items()},
               retention_R=R, confidence_retention_Rc=Rc,
               metrics={f"{a}|{c}": v.to_dict() for (a, c), v in METRICS.items()}, conclusion=TXT)
with open(os.path.join(OUT_DIR, "results_summary.json"), "w") as f:
    json.dump(jsonable(summary), f, indent=2)
print("Saved to", OUT_DIR, ":", sorted(f for f in os.listdir(OUT_DIR) if f != "runs"))

In [ ]:
# ---- Record the software/hardware environment used for this run (for the report and the repository) ----
import sys, platform, subprocess, sklearn, scipy, matplotlib
ENV_USED = dict(python=sys.version.split()[0], platform=platform.platform(), torch=torch.__version__,
                torchvision=torchvision.__version__, numpy=np.__version__, pandas=pd.__version__,
                scikit_learn=sklearn.__version__, scipy=scipy.__version__, matplotlib=matplotlib.__version__,
                cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version() if torch.cuda.is_available() else None,
                device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", amp=USE_AMP)
with open(os.path.join(OUT_DIR, "environment_used.json"), "w") as f:
    json.dump(ENV_USED, f, indent=2)
with open(os.path.join(OUT_DIR, "pip_freeze.txt"), "w") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print(ENV_USED)